# 03 - Preprocessing & Analisis Sentimen (IndoBERT)

**SMATIC 7.0 - Pipeline Tahap 3**

Notebook ini menjalankan dua tahap utama:
1. **Preprocessing** — membersihkan teks dari noise, normalisasi slang, stemming
2. **Analisis Sentimen** — klasifikasi Positif/Negatif/Netral menggunakan IndoBERT

Model yang dipakai: `mdhugol/indonesia-bert-sentiment-classification`, sebuah model klasifikasi sentimen Bahasa Indonesia berbasis BERT.

Input: `analysis_ready_tweets_jakarta_banten.csv` (jumlah baris dibaca secara dinamis)  
Output: `sentiment_results.csv` (dengan kolom sentimen dan confidence score)

## 1. Setup & Load Data

In [ ]:
import pandas as pd
import numpy as np
import re
import json
from pathlib import Path
import warnings

warnings.filterwarnings('default')

from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

INPUT_PATH = Path('analysis_ready_tweets_jakarta_banten.csv')
REQUIRED_COLUMNS = {
    'id', 'text', '_region', '_search_keyword', 'account_class', 'source_weight', 'createdAt'
}

if not INPUT_PATH.exists():
    raise FileNotFoundError(f'Input tidak ditemukan: {INPUT_PATH}. Jalankan notebook 02 terlebih dahulu.')

df = pd.read_csv(INPUT_PATH)
missing_columns = sorted(REQUIRED_COLUMNS - set(df.columns))
if missing_columns:
    raise ValueError(f'Kolom wajib belum tersedia: {missing_columns}')

df['source_weight'] = pd.to_numeric(df['source_weight'], errors='coerce')
if df['source_weight'].isna().any() or (df['source_weight'] <= 0).any():
    raise ValueError('source_weight harus numerik dan lebih besar dari 0 untuk seluruh data analysis-ready.')
print(f'Total tweet: {len(df)}')
print(f'Distribusi region: {dict(df["_region"].value_counts())}')
print(f'Distribusi account_class: {dict(df["account_class"].value_counts())}')
print(f'Duplikat ID: {df["id"].duplicated().sum():,}')

## 2. Preprocessing Pipeline

**Catatan penting:** IndoBERT bekerja lebih baik dengan teks yang tidak di-stem.
Stemming menghilangkan konteks morfologis yang penting bagi model Transformer.
Pipeline ini membuat DUA versi:
- `text_clean` → untuk IndoBERT (tanpa stemming, dengan normalisasi slang)
- `text_stemmed` → untuk LDA Topic Modeling di notebook berikutnya (dengan stemming)

In [ ]:
# ============================================
# KAMUS NORMALISASI SLANG INDONESIA
# ============================================
SLANG_DICT = {
    # Umum
    'gw': 'saya', 'gue': 'saya', 'aku': 'saya', 'w': 'saya',
    'lo': 'kamu', 'lu': 'kamu', 'elo': 'kamu',
    'ga': 'tidak', 'gak': 'tidak', 'ngga': 'tidak', 'enggak': 'tidak',
    'nggak': 'tidak', 'gakbisa': 'tidak bisa', 'gabisa': 'tidak bisa',
    'yg': 'yang', 'dgn': 'dengan', 'utk': 'untuk', 'krn': 'karena',
    'tdk': 'tidak', 'sdh': 'sudah', 'blm': 'belum',
    'jg': 'juga', 'jgn': 'jangan', 'org': 'orang', 'sm': 'sama',
    'tp': 'tapi', 'tpi': 'tapi', 'ttg': 'tentang', 'dr': 'dari',
    'pd': 'pada', 'bs': 'bisa', 'bgt': 'banget', 'bngt': 'banget',
    'bangettt': 'banget', 'sgt': 'sangat', 'skrg': 'sekarang',
    'trs': 'terus', 'trrs': 'terus', 'jd': 'jadi', 'bkn': 'bukan',
    'kyk': 'kayak', 'kek': 'kayak', 'emg': 'memang', 'emang': 'memang',
    'nih': 'ini', 'tuh': 'itu', 'deh': 'deh', 'dong': 'dong', 'sih': 'sih',
    'lah': 'lah', 'kah': 'kah', 'kan': 'kan', 'kok': 'kok',
    # Pendidikan spesifik
    'ppdb': 'PPDB', 'kjp': 'KJP', 'kjmu': 'KJMU', 'ukt': 'UKT',
    'bos': 'BOS', 'snbt': 'SNBT', 'utbk': 'UTBK', 'snbp': 'SNBP',
    'sma': 'SMA', 'smp': 'SMP', 'sd': 'SD',
    # Sentimen khas medsos
    'mantep': 'bagus', 'mantap': 'bagus', 'keren': 'bagus',
    'mager': 'malas', 'males': 'malas', 'ribet': 'rumit',
    'susah': 'sulit', 'payah': 'buruk', 'parah': 'parah',
    'kesel': 'kesal', 'sebel': 'kesal', 'nyesel': 'menyesal',
    'capek': 'lelah', 'capek2': 'lelah', 'hopeless': 'putus asa',
    'zonk': 'gagal', 'zonasi': 'zonasi',
}

print(f'Kamus slang: {len(SLANG_DICT)} entri')

In [ ]:
# ============================================
# FUNGSI PREPROCESSING
# ============================================

def remove_noise(text, keep_numbers=True, keep_sentiment_punctuation=True, keep_mentions=True):
    """Hapus noise teknis sambil mempertahankan informasi penting sesuai tujuan analisis."""
    text = str(text)
    text = re.sub(r'http\S+|www\S+', '', text)          # URL
    if keep_mentions:
        text = re.sub(r'@(?=\w)', '', text)              # simpan nama akun tanpa simbol @
    else:
        text = re.sub(r'@\w+', '', text)                 # buang mention untuk LDA
    text = re.sub(r'#(?=\w)', '', text)                  # simpan kata pada hashtag
    text = re.sub(r'(:-?\)|;-?\)|<3)', ' EMO_POS ', text)
    text = re.sub(r'(:-?\(|T_T)', ' EMO_NEG ', text, flags=re.IGNORECASE)
    text = re.sub(r'[😀😃😄😁😊😍🥰👍❤❤️✨🎉]', ' EMO_POS ', text)
    text = re.sub(r'[😞😔😢😭😡🤬👎💔😩😫]', ' EMO_NEG ', text)
    punctuation_pattern = r'[^\w\s!?]' if keep_sentiment_punctuation else r'[^\w\s]'
    text = re.sub(punctuation_pattern, ' ', text)
    if not keep_numbers:
        text = re.sub(r'(?<!\w)\d+(?!\w)', ' ', text)
    # Emoji lain yang belum dipetakan dibuang agar tokenizer tetap stabil.
    text = re.sub(r'[\U00010000-\U0010FFFF]', '', text, flags=re.UNICODE)
    text = re.sub(r'[\u2600-\u27BF]', '', text)          # simbol misc
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)          # huruf berulang: baguuuus -> baguus
    text = re.sub(r'\s+', ' ', text).strip()             # whitespace
    return text

def normalize_slang(text):
    """Normalisasi kata slang dan singkatan."""
    words = text.lower().split()
    normalized = [SLANG_DICT.get(w, w) for w in words]
    # Filter token kosong (dari penghapusan partikel)
    normalized = [w for w in normalized if w.strip()]
    return ' '.join(normalized)

SPAM_PATTERNS = {
    'job_vacancy': r'\b(lowongan|loker|rekrutmen)\b|dibutuhkan.{0,30}\b(posisi|karyawan)\b',
    'commercial': r'\b(promo|diskon)\b.{0,50}\b(kode|voucher|checkout|link)\b|\b(jual|beli)\b.{0,30}\b(dm|wa|order)\b|\b(rental|sewa)\b.{0,30}\b(mobil|motor|kost|kos)\b',
    'engagement_bait': r'follow.{0,15}back|\brt\b.{0,30}\b(menang|hadiah|giveaway)\b',
}

def detect_spam_reason(text):
    """Kembalikan alasan spam agar setiap penghapusan bisa diaudit."""
    text_lower = str(text).lower()
    for reason, pattern in SPAM_PATTERNS.items():
        if re.search(pattern, text_lower):
            return reason
    return ''

# Inisialisasi Sastrawi
stemmer = StemmerFactory().create_stemmer()
stopword_remover = StopWordRemoverFactory().create_stop_word_remover()

# Nama wilayah dan istilah kebijakan tidak boleh berubah akibat stemming.
PROTECTED_WORDS = {
    'tangerang selatan', 'jakarta', 'banten', 'bekasi', 'tangerang',
    'serang', 'pandeglang', 'lebak', 'cilegon', 'depok', 'bogor',
    'ppdb', 'spmb', 'kjp', 'kjmu', 'ukt', 'bos', 'snbt', 'snbp', 'utbk',
}

def protect_terms(text):
    protected = {}
    # Dahulukan frasa panjang agar 'tangerang selatan' tidak terpotong.
    for index, term in enumerate(sorted(PROTECTED_WORDS, key=len, reverse=True)):
        placeholder = f'zzprotected{index}zz'
        updated = re.sub(rf'\b{re.escape(term)}\b', placeholder, text, flags=re.IGNORECASE)
        if updated != text:
            protected[placeholder] = term
            text = updated
    return text, protected

def restore_terms(text, protected):
    for placeholder, term in protected.items():
        text = re.sub(rf'\b{placeholder}\b', term, text, flags=re.IGNORECASE)
    return text

def preprocess_for_bert(text):
    """Pipeline untuk IndoBERT: noise removal + normalisasi slang (tanpa stemming)."""
    text = remove_noise(
        text, keep_numbers=True, keep_sentiment_punctuation=True, keep_mentions=True
    )
    text = normalize_slang(text)
    return text.strip()

def preprocess_for_lda(text):
    """Pipeline untuk LDA: full preprocessing termasuk stopword + stemming."""
    text = remove_noise(
        text, keep_numbers=False, keep_sentiment_punctuation=False, keep_mentions=False
    )
    text = normalize_slang(text)
    text = re.sub(r'\bemo_(pos|neg)\b', ' ', text)
    text, protected = protect_terms(text)
    text = stopword_remover.remove(text)
    text = stemmer.stem(text)
    text = restore_terms(text, protected)
    return text.lower().strip()

print('Fungsi preprocessing siap.')

In [ ]:
# ============================================
# JALANKAN PREPROCESSING
# ============================================

print('Filtering spam content...')
df['_spam_reason'] = df['text'].apply(detect_spam_reason)
df['_is_spam_content'] = df['_spam_reason'].ne('')
n_spam = df['_is_spam_content'].sum()
print(f'Tweet dibuang karena konten spam/tidak relevan: {n_spam}')
if n_spam:
    print(df.loc[df['_is_spam_content'], '_spam_reason'].value_counts().to_string())

spam_audit = df[df['_is_spam_content']].copy()
df_clean = df[~df['_is_spam_content']].copy()
print(f'Tweet tersisa setelah filter konten: {len(df_clean)}')

print('\nMenjalankan preprocessing untuk IndoBERT...')
df_clean['text_clean'] = df_clean['text'].apply(preprocess_for_bert)

print('Menjalankan preprocessing untuk LDA (lebih lambat karena stemming)...')
df_clean['text_stemmed'] = df_clean['text'].apply(preprocess_for_lda)

# Filter tweet yang terlalu pendek setelah preprocessing.
df_clean['_text_clean_len'] = df_clean['text_clean'].str.len()
short_text_mask = df_clean['_text_clean_len'] < 10
short_text_audit = df_clean[short_text_mask].copy()
df_clean = df_clean[~short_text_mask].copy()
print(f'\nTweet terlalu pendek setelah preprocessing (< 10 karakter): {short_text_mask.sum()}')

# ID sudah didedup pada notebook 01; tahap ini menangani salinan teks dari akun berbeda.
df_clean['_is_exact_text_duplicate'] = df_clean.duplicated('text_clean', keep='first')
duplicate_text_audit = df_clean[df_clean['_is_exact_text_duplicate']].copy()
df_clean = df_clean[~df_clean['_is_exact_text_duplicate']].copy()
print(f'Duplikat teks identik dibuang: {len(duplicate_text_audit)}')
print(f'Total final untuk analisis sentimen: {len(df_clean)}')

In [ ]:
# Cek hasil preprocessing - bandingkan teks asli vs bersih
print('=== SAMPLE PERBANDINGAN TEKS ===')
for i, row in df_clean[['text', 'text_clean', 'text_stemmed']].head(5).iterrows():
    print(f'ASLI   : {str(row["text"])[:120]}')
    print(f'BERSIH : {str(row["text_clean"])[:120]}')
    print(f'STEMMED: {str(row["text_stemmed"])[:120]}')
    print()

## 3. Analisis Sentimen dengan IndoBERT

Model: `mdhugol/indonesia-bert-sentiment-classification`
- Label resmi model: `LABEL_0` = Positif, `LABEL_1` = Netral, `LABEL_2` = Negatif
- Input max 512 token — tweet yang lebih panjang akan di-truncate otomatis
- **Catatan:** model ini di-download otomatis (~440MB) dari HuggingFace saat pertama kali dijalankan

In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
import torch

MODEL_NAME = 'mdhugol/indonesia-bert-sentiment-classification'
MODEL_REVISION = '80ccb4c2817cf976534ac491020a9572e5dae54f'

print(f'Loading model: {MODEL_NAME}')
print('(Download ~440MB dari HuggingFace jika pertama kali — tunggu beberapa menit)')

device = 0 if torch.cuda.is_available() else -1
print(f'Device: {"GPU" if device == 0 else "CPU"}')

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME, revision=MODEL_REVISION, token=False
)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, revision=MODEL_REVISION, token=False
)
model_revision = getattr(model.config, '_commit_hash', None) or 'unknown'

# Mapping resmi dari model card: LABEL_0=positive, LABEL_1=neutral, LABEL_2=negative.
LABEL_MAP = {
    'LABEL_0': 'Positif', 'positive': 'Positif', 'positif': 'Positif',
    'LABEL_1': 'Netral', 'neutral': 'Netral', 'netral': 'Netral',
    'LABEL_2': 'Negatif', 'negative': 'Negatif', 'negatif': 'Negatif',
}
configured_labels = [str(v) for _, v in sorted(model.config.id2label.items())]
unsupported_labels = [label for label in configured_labels if label not in LABEL_MAP and label.lower() not in LABEL_MAP]
if unsupported_labels:
    raise ValueError(f'Label model belum dikenali: {unsupported_labels}. Periksa model.config.id2label.')

sentiment_pipeline = pipeline(
    'text-classification',
    model=model,
    tokenizer=tokenizer,
    device=device,
    truncation=True,
    max_length=512,
)

def map_model_label(label):
    return LABEL_MAP.get(label, LABEL_MAP.get(str(label).lower()))

print(f'Model berhasil di-load. Revision: {model_revision}')
print(f'Label config: {model.config.id2label}')

In [ ]:
# Test model dengan beberapa contoh manual
test_texts = [
    'sistem zonasi ini sangat tidak adil dan menyusahkan orang tua',
    'alhamdulillah beasiswa Banten sudah cair bulan ini',
    'PPDB Jakarta dibuka mulai bulan depan',
    'biaya kuliah semakin tidak terjangkau bagi keluarga tidak mampu',
    'pemerintah sudah berusaha meningkatkan akses pendidikan di daerah terpencil',
]

print('=== TEST MODEL SENTIMEN ===')
results = sentiment_pipeline(test_texts)
for text, result in zip(test_texts, results):
    label = map_model_label(result['label'])
    score = result['score']
    print(f'{label:10} ({score:.3f}) | {text[:80]}')

In [ ]:
# ============================================
# INFERENCE - SEMUA TWEET
# ============================================
# Jalankan dalam batch untuk efisiensi

BATCH_SIZE = 32
texts = df_clean['text_clean'].tolist()

print(f'Menjalankan analisis sentimen untuk {len(texts)} tweet...')
print(f'Batch size: {BATCH_SIZE}')
print('Estimasi waktu: 2-5 menit di CPU, <1 menit di GPU')

all_results = []
for i in range(0, len(texts), BATCH_SIZE):
    batch = texts[i:i+BATCH_SIZE]
    batch_results = sentiment_pipeline(batch, top_k=None)
    all_results.extend(batch_results)
    
    if (i // BATCH_SIZE) % 10 == 0:
        print(f'  Progress: {i+len(batch)}/{len(texts)} ({(i+len(batch))/len(texts)*100:.1f}%)')

def unpack_class_scores(result):
    if isinstance(result, dict):
        result = [result]
    return {map_model_label(item['label']): float(item['score']) for item in result}

class_scores = [unpack_class_scores(result) for result in all_results]
required_sentiments = {'Negatif', 'Netral', 'Positif'}
if any(set(scores) != required_sentiments for scores in class_scores):
    raise ValueError('Output model tidak memuat tepat tiga kelas sentimen yang diharapkan.')

df_clean['prob_negatif'] = [scores['Negatif'] for scores in class_scores]
df_clean['prob_netral'] = [scores['Netral'] for scores in class_scores]
df_clean['prob_positif'] = [scores['Positif'] for scores in class_scores]
df_clean['sentiment_label'] = [max(scores, key=scores.get) for scores in class_scores]
df_clean['sentiment_score'] = [max(scores.values()) for scores in class_scores]
df_clean['sentiment_margin'] = [
    sorted(scores.values(), reverse=True)[0] - sorted(scores.values(), reverse=True)[1]
    for scores in class_scores
]
df_clean['sentiment_entropy'] = [
    -sum(p * np.log(max(p, 1e-12)) for p in scores.values()) / np.log(3)
    for scores in class_scores
]

CONFIDENCE_THRESHOLD = 0.60
MARGIN_THRESHOLD = 0.15
df_clean['sentiment_review_status'] = np.where(
    (df_clean['sentiment_score'] < CONFIDENCE_THRESHOLD) |
    (df_clean['sentiment_margin'] < MARGIN_THRESHOLD),
    'needs_manual_review',
    'accepted',
)

df_clean['sentiment_value'] = df_clean['sentiment_label'].map(
    {'Positif': 1, 'Netral': 0, 'Negatif': -1}
)
df_clean['weighted_sentiment_numerator'] = df_clean['sentiment_value'] * df_clean['source_weight']

print('\nSelesai! Distribusi sentimen:')
print(df_clean['sentiment_label'].value_counts())
print(f'\nRata-rata confidence score: {df_clean["sentiment_score"].mean():.3f}')
print(f'Perlu review manual: {(df_clean["sentiment_review_status"] == "needs_manual_review").sum():,}')

## 4. Analisis Hasil per Wilayah

In [ ]:
print('=== DISTRIBUSI SENTIMEN PER WILAYAH ===')
sentiment_by_region = pd.crosstab(
    df_clean['_region'],
    df_clean['sentiment_label'],
    normalize='index'
) * 100
print(sentiment_by_region.round(2))

def weighted_sentiment_score(group):
    total_weight = group['source_weight'].sum()
    if total_weight <= 0:
        return np.nan
    return group['weighted_sentiment_numerator'].sum() / total_weight

def weighted_sentiment_distribution(data, index_column):
    weighted_counts = (
        data.groupby([index_column, 'sentiment_label'])['source_weight']
        .sum()
        .unstack(fill_value=0)
    )
    return weighted_counts.div(weighted_counts.sum(axis=1), axis=0) * 100

print('\n=== DISTRIBUSI SENTIMEN BERBOBOT PER WILAYAH ===')
print(weighted_sentiment_distribution(df_clean, '_region').round(2))

print('\n=== SKOR SENTIMEN RATA-RATA PER WILAYAH (weighted) ===')
# -1 = sangat negatif, +1 = sangat positif
score_by_region = (
    df_clean.groupby('_region')[['weighted_sentiment_numerator', 'source_weight']]
    .apply(weighted_sentiment_score)
)
for region, score in score_by_region.items():
    label = '(cenderung negatif)' if score < -0.1 else '(cenderung positif)' if score > 0.1 else '(netral)'
    print(f'  {region}: {score:.3f} {label}')

accepted_only = df_clean[df_clean['sentiment_review_status'] == 'accepted']
accepted_score_by_region = (
    accepted_only.groupby('_region')[['weighted_sentiment_numerator', 'source_weight']]
    .apply(weighted_sentiment_score)
)
print('\nSensitivity check (hanya prediksi accepted):')
print(accepted_score_by_region.round(3).to_string())

print('\n=== DISTRIBUSI SENTIMEN PER KEYWORD ===')
sentiment_by_keyword = pd.crosstab(
    df_clean['_search_keyword'],
    df_clean['sentiment_label'],
    normalize='index'
) * 100
print(sentiment_by_keyword.round(1))

print('\n=== DISTRIBUSI SENTIMEN BERBOBOT PER KEYWORD ===')
print(weighted_sentiment_distribution(df_clean, '_search_keyword').round(1))

In [ ]:
print('=== DISTRIBUSI SENTIMEN PER ACCOUNT CLASS ===')
print(pd.crosstab(
    df_clean['account_class'],
    df_clean['sentiment_label'],
    normalize='index'
).round(3) * 100)

print('\n=== TWEET NEGATIF TERTINGGI CONFIDENCE SCORE ===')
top_negative = df_clean[
    df_clean['sentiment_label'] == 'Negatif'
].nlargest(10, 'sentiment_score')[['text', '_region', 'sentiment_score', 'account_class']]

pd.set_option('display.max_colwidth', 100)
for _, row in top_negative.iterrows():
    print(f'[{row["_region"]}][score:{row["sentiment_score"]:.3f}] {str(row["text"])[:120]}')

print('\n=== TWEET POSITIF TERTINGGI CONFIDENCE SCORE ===')
top_positive = df_clean[
    df_clean['sentiment_label'] == 'Positif'
].nlargest(10, 'sentiment_score')[['text', '_region', 'sentiment_score']]

for _, row in top_positive.iterrows():
    print(f'[{row["_region"]}][score:{row["sentiment_score"]:.3f}] {str(row["text"])[:120]}')

# ============================================
# EVALUASI MANUAL (WAJIB UNTUK HASIL FINAL)
# ============================================
MANUAL_LABEL_PATH = Path('manual_validation_sample.csv')
VALID_SENTIMENTS = {'Negatif', 'Netral', 'Positif'}

if MANUAL_LABEL_PATH.exists():
    manual_labels = pd.read_csv(MANUAL_LABEL_PATH, dtype={'id': str})
    if not {'id', 'manual_sentiment_label'}.issubset(manual_labels.columns):
        raise ValueError('manual_validation_sample.csv wajib memiliki kolom id dan manual_sentiment_label.')

    manual_labels['manual_sentiment_label'] = manual_labels['manual_sentiment_label'].fillna('').str.strip().str.title()
    invalid_manual = set(manual_labels['manual_sentiment_label']) - VALID_SENTIMENTS - {''}
    if invalid_manual:
        raise ValueError(f'Label manual tidak valid: {sorted(invalid_manual)}')

    evaluation = df_clean.assign(id=df_clean['id'].astype(str)).merge(
        manual_labels[['id', 'manual_sentiment_label']], on='id', how='inner'
    )
    evaluation = evaluation[evaluation['manual_sentiment_label'].isin(VALID_SENTIMENTS)]

    if len(evaluation) >= 30:
        from sklearn.metrics import classification_report, confusion_matrix

        label_order = ['Negatif', 'Netral', 'Positif']
        print(f'\n=== EVALUASI MANUAL ({len(evaluation)} tweet) ===')
        print(classification_report(
            evaluation['manual_sentiment_label'], evaluation['sentiment_label'],
            labels=label_order, digits=3, zero_division=0,
        ))
        print('Confusion matrix (aktual x prediksi):')
        print(pd.DataFrame(
            confusion_matrix(
                evaluation['manual_sentiment_label'], evaluation['sentiment_label'], labels=label_order
            ),
            index=[f'aktual_{x}' for x in label_order],
            columns=[f'pred_{x}' for x in label_order],
        ))
    else:
        print(f'Label manual terisi baru {len(evaluation)}; isi minimal 30, idealnya 150-300 tweet.')
else:
    sample_size = min(200, len(df_clean))
    validation_sample = (
        df_clean.groupby(['_region', 'sentiment_label', 'sentiment_review_status'], group_keys=False)
        .apply(lambda group: group.sample(min(len(group), 10), random_state=42))
        .drop_duplicates('id')
    )
    remaining = df_clean[~df_clean['id'].isin(validation_sample['id'])]
    if len(validation_sample) < sample_size:
        validation_sample = pd.concat([
            validation_sample,
            remaining.sample(min(sample_size - len(validation_sample), len(remaining)), random_state=42),
        ])
    validation_sample = validation_sample.head(sample_size).copy()
    validation_sample['manual_sentiment_label'] = ''
    # Prediksi model sengaja tidak disertakan agar anotator tidak terpengaruh hasil model.
    validation_sample[[
        'id', 'text', '_region', '_search_keyword', 'manual_sentiment_label',
    ]].to_csv(MANUAL_LABEL_PATH, index=False)
    print(f'\nTemplate evaluasi manual dibuat: {MANUAL_LABEL_PATH} ({len(validation_sample)} tweet)')

## 5. Analisis Tren Temporal

In [ ]:
df_clean['createdAt'] = pd.to_datetime(df_clean['createdAt'], errors='coerce', utc=True)
df_clean['year_month'] = df_clean['createdAt'].dt.tz_convert(None).dt.to_period('M')

valid_dates = df_clean.dropna(subset=['year_month'])

print('=== TREN SENTIMEN PER BULAN ===')
trend = valid_dates.groupby(['year_month', 'sentiment_label']).size().unstack(fill_value=0)
trend['total'] = trend.sum(axis=1)

if 'Negatif' in trend.columns:
    trend['pct_negatif'] = (trend['Negatif'] / trend['total'] * 100).round(1)

print(trend.tail(12).to_string())

print('\n=== TREN PER BULAN PER WILAYAH ===')
trend_region = valid_dates.groupby(
    ['year_month', '_region', 'sentiment_label']
).size().unstack(fill_value=0)
print(trend_region.tail(24).to_string())

print('\n=== TREN SENTIMEN BERBOBOT PER BULAN ===')
weighted_trend = (
    valid_dates.groupby(['year_month', 'sentiment_label'])['source_weight']
    .sum()
    .unstack(fill_value=0)
)
weighted_trend_pct = weighted_trend.div(weighted_trend.sum(axis=1), axis=0) * 100
monthly_weighted_score = (
    valid_dates.groupby('year_month')[['weighted_sentiment_numerator', 'source_weight']]
    .apply(weighted_sentiment_score)
    .rename('weighted_score')
)
print(weighted_trend_pct.tail(12).round(1).to_string())
print('\nSkor bulanan berbobot:')
print(monthly_weighted_score.tail(12).round(3).to_string())

## 6. Simpan Hasil

In [ ]:
# Simpan dataset lengkap dengan label sentimen
sentiment_output_dir = Path('organized_csv/05_sentiment')
audit_output_dir = sentiment_output_dir / 'audits'
audit_output_dir.mkdir(parents=True, exist_ok=True)

df_clean.to_csv('sentiment_results.csv', index=False)
df_clean.to_csv(sentiment_output_dir / 'sentiment_results.csv', index=False)
print(f'Saved: sentiment_results.csv ({len(df_clean)} rows)')
spam_audit.to_csv(audit_output_dir / 'excluded_spam.csv', index=False)
short_text_audit.to_csv(audit_output_dir / 'excluded_short_text.csv', index=False)
duplicate_text_audit.to_csv(audit_output_dir / 'excluded_exact_duplicates.csv', index=False)
df_clean[df_clean['sentiment_review_status'] == 'needs_manual_review'].to_csv(
    audit_output_dir / 'low_confidence_predictions.csv', index=False
)

# Simpan ringkasan per wilayah untuk dashboard
summary_region = df_clean.groupby('_region').agg(
    n_tweets=('id', 'count'),
    n_positif=('sentiment_label', lambda x: (x=='Positif').sum()),
    n_netral=('sentiment_label', lambda x: (x=='Netral').sum()),
    n_negatif=('sentiment_label', lambda x: (x=='Negatif').sum()),
    mean_confidence=('sentiment_score', 'mean'),
    n_needs_review=('sentiment_review_status', lambda x: (x=='needs_manual_review').sum()),
    total_source_weight=('source_weight', 'sum'),
).reset_index()
summary_region['weighted_score'] = summary_region['_region'].map(score_by_region)
summary_region['weighted_score_accepted_only'] = summary_region['_region'].map(accepted_score_by_region)

summary_region['pct_positif'] = (summary_region['n_positif'] / summary_region['n_tweets'] * 100).round(1)
summary_region['pct_negatif'] = (summary_region['n_negatif'] / summary_region['n_tweets'] * 100).round(1)
summary_region['pct_netral'] = (summary_region['n_netral'] / summary_region['n_tweets'] * 100).round(1)

weighted_region_pct = weighted_sentiment_distribution(df_clean, '_region').add_prefix('weighted_pct_').reset_index()
summary_region = summary_region.merge(weighted_region_pct, on='_region', how='left')

summary_region.to_csv('summary_by_region.csv', index=False)
summary_region.to_csv(sentiment_output_dir / 'summary_by_region.csv', index=False)
print(f'Saved: summary_by_region.csv')
print()
print(summary_region.to_string())

# Simpan ringkasan per keyword untuk analisis isu
summary_keyword = df_clean.groupby(['_search_keyword', '_region', 'sentiment_label']).size().reset_index(name='count')
summary_keyword.to_csv('summary_by_keyword.csv', index=False)
summary_keyword.to_csv(sentiment_output_dir / 'summary_by_keyword.csv', index=False)
weighted_keyword_summary = weighted_sentiment_distribution(df_clean, '_search_keyword').reset_index()
score_by_keyword = (
    df_clean.groupby('_search_keyword')[['weighted_sentiment_numerator', 'source_weight']]
    .apply(weighted_sentiment_score)
    .rename('weighted_score')
    .reset_index()
)
weighted_keyword_summary = weighted_keyword_summary.merge(score_by_keyword, on='_search_keyword', how='left')
weighted_keyword_summary.to_csv('weighted_summary_by_keyword.csv', index=False)
weighted_keyword_summary.to_csv(sentiment_output_dir / 'weighted_summary_by_keyword.csv', index=False)
print(f'\nSaved: summary_by_keyword.csv')
trend.to_csv(sentiment_output_dir / 'monthly_sentiment_counts.csv')
weighted_trend_pct.to_csv(sentiment_output_dir / 'monthly_weighted_sentiment_pct.csv')
monthly_weighted_score.to_csv(sentiment_output_dir / 'monthly_weighted_score.csv')

# Simpan teks stemmed untuk LDA (notebook berikutnya)
df_clean[['id', 'text', 'text_clean', 'text_stemmed', '_region', 
          '_search_keyword', 'account_class', 'sentiment_label', 
          'sentiment_score', 'sentiment_review_status', 'source_weight', 'createdAt', 'year_month']].to_csv(
    'data_for_lda.csv', index=False
)
df_clean[['id', 'text', 'text_clean', 'text_stemmed', '_region',
          '_search_keyword', 'account_class', 'sentiment_label',
          'sentiment_score', 'sentiment_review_status', 'source_weight', 'createdAt', 'year_month']].to_csv(
    sentiment_output_dir / 'data_for_lda.csv', index=False
)

run_metadata = {
    'model_name': MODEL_NAME,
    'model_revision': model_revision,
    'torch_version': torch.__version__,
    'transformers_version': __import__('transformers').__version__,
    'confidence_threshold': CONFIDENCE_THRESHOLD,
    'margin_threshold': MARGIN_THRESHOLD,
    'n_input': int(len(df)),
    'n_analyzed': int(len(df_clean)),
}
(sentiment_output_dir / 'run_metadata.json').write_text(
    json.dumps(run_metadata, indent=2), encoding='utf-8'
)
print(f'Saved: data_for_lda.csv (untuk notebook LDA Topic Modeling)')
print(f'Output terorganisasi: {sentiment_output_dir}')

## Catatan Metodologis untuk Esai

Bagian ini bisa langsung diadaptasi ke Metodologi esai:

> Analisis sentimen dilakukan menggunakan model *Indonesia BERT Sentiment Classifier*
> (`mdhugol/indonesia-bert-sentiment-classification`), sebuah model berbasis BERT yang
> telah di-*fine-tune* untuk klasifikasi sentimen Bahasa Indonesia pada data Twitter/X.
> Sebelum inferensi, teks melalui tahapan *preprocessing* konservatif: penghapusan URL,
> mempertahankan nama mention dan kata dalam hashtag tanpa simbolnya, mempertahankan angka
> serta tanda tanya/seru, pemetaan emoji positif/negatif, dan normalisasi
> kosakata slang Bahasa Indonesia, serta filter konten spam yang menyimpan alasan eksklusi.
> Stemming sengaja tidak
> diterapkan pada data untuk analisis sentimen karena model Transformer lebih optimal
> bekerja dengan teks yang mempertahankan konteks morfologis aslinya. Model mengklasifikasikan
> setiap tweet ke dalam tiga kelas: Positif, Netral, atau Negatif. Probabilitas seluruh kelas,
> margin antarkelas, dan entropy disimpan sebagai indikator ketidakpastian; prediksi ambigu
> diarahkan ke pemeriksaan manual dan tidak diklaim sebagai reliabilitas terkalibrasi. Skor
> sentimen wilayah dihitung sebagai jumlah skor kali bobot dibagi total bobot sumber. Bobot
> merepresentasikan tipe sumber, bukan ukuran kebenaran suatu pendapat. Kinerja model perlu
> dilaporkan menggunakan sampel berlabel manual secara buta terhadap prediksi model,
> confusion matrix, dan macro-F1. Label wilayah dalam dataset menunjukkan wilayah query
> pengambilan data, bukan bukti domisili pengguna Twitter/X.

## Langkah Selanjutnya

1. Lanjut ke `04_lda_topic_modeling.ipynb` menggunakan `data_for_lda.csv`
2. Lanjut ke `05_demographic_weighting.ipynb` menggunakan `sentiment_results.csv`
3. Setelah keduanya selesai, gabungkan ke `06_dashboard.py` (Streamlit)